# AstroHack SATEM 2026 · Cuaderno inicial
Ejecuta desde la carpeta `herramientas` del paquete descomprimido. No necesitas descargar ni calibrar visibilidades. Elige tus hipótesis: este cuaderno no contiene la solución.

**Objetivo:** dos hipótesis moleculares, tres rasgos por hipótesis cuando existan, y una asignación descartada o declarada insuficiente. La calidad de la evidencia pesa más que el resultado positivo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from astropy.io import fits
from astropy.wcs import WCS
ROOT = Path.cwd()
if not (ROOT / "espectros").exists(): ROOT = ROOT.parent
assert (ROOT / "espectros/W1_espectros.csv").exists(), "Abre desde herramientas o desde la raíz del paquete"
esp = {i: pd.read_csv(ROOT / f"espectros/W{i}_espectros.csv") for i in (1,2)}
cat = pd.read_csv(ROOT / "catalogos/catalogo_local.csv", dtype={"entrada": str})
print("Filas catálogo:", len(cat), "Canales:", [len(esp[i]) for i in (1,2)])

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(12,5),layout="constrained")
for i,ax in enumerate(axes,1):
    ax.plot(esp[i].frecuencia_LSRK_GHz, esp[i].B_offset_Jy_beam, lw=.7)
    ax.set(xlabel="Frecuencia LSRK (GHz)", ylabel="Jy/haz", title=f"W{i} · B_offset")
plt.show()

## Frecuencia y candidatos
Usamos velocidad radio: v = c (1 − ν_LSRK / ν_reposo). Para empezar puede explorarse v ≈ 2.7 km/s en B; debe comprobarse con varias líneas. Esta velocidad no se impone a A. La tolerancia depende de canal, perfil, error de catálogo y mezclas.

Selecciona una frecuencia **observada** de un pico. La función busca frecuencias en reposo cercanas después de corregir Doppler. Una coincidencia no identifica por sí misma una molécula.

In [ ]:
C = 299792.458

def candidatos(nu_observada_GHz, v_kms=2.7, tolerancia_MHz=1.0):
    nu_reposo_MHz = 1000*nu_observada_GHz/(1-v_kms/C)
    d = cat.assign(delta_MHz=cat.frecuencia_reposo_MHz-nu_reposo_MHz)
    return d[abs(d.delta_MHz)<=tolerancia_MHz].sort_values("delta_MHz")

# Completa tras examinar el espectro:
# candidatos(nu_observada_GHz=...)

## Ruido y ajuste local
Selecciona al menos tres intervalos sin rasgos visibles, evita absorciones y bordes. No uses el RMS de toda la ventana, dominado por líneas. Registra tus intervalos. En canales correlacionados, la incertidumbre del área no sigue automáticamente √N.
El ajuste gaussiano es descriptivo: revisa residuos. Un perfil asimétrico o mezclado puede invalidarlo. El error formal no incluye calibración, opacidad, selección de baseline ni resolución instrumental.

In [ ]:
def ruido_empirico(ventana, intervalos_GHz, posicion="B_offset"):
    d=esp[ventana]; f=d.frecuencia_LSRK_GHz.to_numpy(); y=d[posicion+"_Jy_beam"].to_numpy()
    piezas=[]
    for lo,hi in intervalos_GHz:
        z=y[(f>=lo)&(f<=hi)]
        if len(z)<5: raise ValueError("Se requieren al menos 5 canales por intervalo")
        piezas.append(z-np.median(z))
    return np.std(np.concatenate(piezas),ddof=1)

def ajustar(ventana, nu_reposo_GHz, vmin, vmax, posicion="B_offset"):
    d=esp[ventana]; f=d.frecuencia_LSRK_GHz.to_numpy(); y=d[posicion+"_Jy_beam"].to_numpy()
    v=C*(1-f/nu_reposo_GHz); m=(v>=vmin)&(v<=vmax); x,z=v[m],y[m]
    if len(x)<9: raise ValueError("Usa una región con al menos 9 canales")
    def gauss(x,a,mu,sigma,b):return b+a*np.exp(-.5*((x-mu)/sigma)**2)
    base=np.median(z); amp=max(z)-base; mu=x[np.argmax(z)]
    par,cov=curve_fit(gauss,x,z,p0=[amp,mu,.5,base],bounds=([0,vmin,.1,-np.inf],[np.inf,vmax,3,np.inf]))
    a,mu,sd,b=par; residual=z-gauss(x,*par)
    fig,ax=plt.subplots(figsize=(7,3)); ax.step(x,z,where="mid",label="datos")
    xx=np.linspace(vmin,vmax,300);ax.plot(xx,gauss(xx,*par),label="gaussiana + constante")
    ax.plot(x,residual,label="residuos"); ax.set(xlabel="v radio LSRK (km/s)",ylabel="Jy/haz"); ax.legend();plt.show()
    return dict(v_centro=mu,FWHM=2.35482*sd,pico=a,baseline=b,area=a*sd*np.sqrt(2*np.pi),errores_formales=np.sqrt(np.diag(cov)),rms_residuo=np.std(residual,ddof=4))

# ruido_empirico(1, [(inicio_GHz, fin_GHz), ...])
# ajustar(ventana=1, nu_reposo_GHz=..., vmin=..., vmax=...)

## Cubo opcional y extracción verificable
Los índices son de Python, empiezan en cero. B_offset está en (x,y)=(54,46) en ambos recortes. El valor es Jy por haz en un píxel; sumar píxeles no da directamente Jy. No uses RESTFRQ de una ventana como frecuencia de todas las transiciones.
Para comparar W1/W2 espacialmente, ten en cuenta que los haces difieren. Para integrar emisión, conserva el eje de frecuencia y convierte Δν a Δv por transición.

In [ ]:
with fits.open(ROOT / "cubos/W1_IRAS16293.fits", memmap=True) as hd:
    cube=hd[0].data
    header=hd[0].header
    print(cube.shape, header["BUNIT"], header["SPECSYS"])
    extraido=cube[:,46,54].copy()
assert np.allclose(extraido,esp[1].B_offset_Jy_beam,rtol=1e-7,atol=1e-9)
print("Extracción B_offset verificada")

## Entrega
Rellena `evidencia_plantilla.csv`. Agrupa transiciones no resueltas en un único rasgo. Incluye frecuencias observadas y de reposo, errores, alternativas, mezclas y decisiones. Exporta figuras legibles; añade un README que permita repetir tus resultados. Declara uso de IA, catálogos y bibliografía. Para una conclusión insuficiente, explica qué medición adicional la resolvería.